GitHub<br>
https://github.com/Vegetebird/StridedTransformer-Pose3D<br>
論文<br>
https://arxiv.org/abs/2103.14304<br>
<br>
<a href="https://colab.research.google.com/github/kaz12tech/ai_demos/blob/master/StridedTransformer_Pose3D.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 環境セットアップ

## GPU確認

In [1]:
!nvidia-smi

Sun Aug 25 11:43:09 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              10W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

## Githubからソースコード取得

In [2]:
%cd /content

!git clone https://github.com/Vegetebird/StridedTransformer-Pose3D.git

/content
Cloning into 'StridedTransformer-Pose3D'...
remote: Enumerating objects: 229, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 229 (delta 38), reused 50 (delta 26), pack-reused 165 (from 1)
Receiving objects: 100% (229/229), 28.37 MiB | 19.21 MiB/s, done.
Resolving deltas: 100% (96/96), done.


## ライブラリのインストール

In [3]:
%cd /content/StridedTransformer-Pose3D

!pip install --upgrade gdown
!pip install yacs
!pip install filterpy
!pip install einops
!pip install yt-dlp moviepy
#!pip install matplotlib==3.0.3
#!pip install imageio==2.4.1

/content/StridedTransformer-Pose3D
  Attempting uninstall: gdown
    Found existing installation: gdown 5.1.0
    Uninstalling gdown-5.1.0:
      Successfully uninstalled gdown-5.1.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110459 sha256=acd5f484aed2c00f78d2bce2330977b81ce0d65a5e73a9496ebc25cfc455706f
  Stored in directory: /root/.cache/pip/wheels/0f/0c/ea/218f266af4ad626897562199fbbcba521b8497303200186102
Successfully built filterpy
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.2/157.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.4/194.4 kB 15.1 MB/s eta 0:00:00
   ━━━

## ライブラリのインポート

In [4]:
%cd /content/StridedTransformer-Pose3D

import os
from yt_dlp import YoutubeDL

from moviepy.video.fx.resize import resize
from moviepy.editor import VideoFileClip, AudioFileClip, ImageSequenceClip, CompositeAudioClip
from moviepy.video.io.ffmpeg_tools import ffmpeg_extract_subclip

/content/StridedTransformer-Pose3D


# 学習済みモデルのセットアップ

In [5]:
%cd /content/StridedTransformer-Pose3D
!mkdir -p ./checkpoint/pretrained

if not os.path.exists('checkpoint/pretrained/refine_4365.pth'):
  !gdown https://drive.google.com/uc?id=1aDLu0SB9JnPYZOOzQsJMV9zEIHg2Uro7 -O checkpoint/pretrained/refine_4365.pth
if not os.path.exists('checkpoint/pretrained/no_refine_4365.pth'):
  !gdown https://drive.google.com/uc?id=1l63AI9BsNovpfTAbfAkySo9X2MOWgYZH -O checkpoint/pretrained/no_refine_4365.pth

if not os.path.exists('demo/lib/checkpoint/yolov3.weights'):
  !gdown https://drive.google.com/uc?id=1gWZl1VrlLZKBf0Pfkj4hKiFxe8sHP-1C -O demo/lib/checkpoint/yolov3.weights
if not os.path.exists('demo/lib/checkpoint/pose_hrnet_w48_384x288.pth'):
  !gdown https://drive.google.com/uc?id=1CpyZiUIUlEjiql4rILwdBT4666S72Oq4 -O demo/lib/checkpoint/pose_hrnet_w48_384x288.pth

/content/StridedTransformer-Pose3D
Downloading...
From: https://drive.google.com/uc?id=1aDLu0SB9JnPYZOOzQsJMV9zEIHg2Uro7
To: /content/StridedTransformer-Pose3D/checkpoint/pretrained/refine_4365.pth
100% 563k/563k [00:00<00:00, 40.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1l63AI9BsNovpfTAbfAkySo9X2MOWgYZH
To: /content/StridedTransformer-Pose3D/checkpoint/pretrained/no_refine_4365.pth
100% 17.4M/17.4M [00:00<00:00, 67.2MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1gWZl1VrlLZKBf0Pfkj4hKiFxe8sHP-1C
From (redirected): https://drive.google.com/uc?id=1gWZl1VrlLZKBf0Pfkj4hKiFxe8sHP-1C&confirm=t&uuid=9132c751-ee77-4c9c-96f5-af6f588d6886
To: /content/StridedTransformer-Pose3D/demo/lib/checkpoint/yolov3.weights
100% 248M/248M [00:01<00:00, 143MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1CpyZiUIUlEjiql4rILwdBT4666S72Oq4
From (redirected): https://drive.google.com/uc?id=1CpyZiUIUlEjiql4rILwdBT4666S72Oq4&confirm=t&uuid=c1e64405-7d1c-4

# `独自`





In [7]:

#!python demo/vis.py
!python demo/vis.py --video ch.mp4
#/content/StridedTransformer-Pose3D/demo/video/IMG1837.mp4




Generating 2D pose...
100% 309/309 [00:34<00:00,  8.85it/s]
Generating 2D pose successful!

Generating 3D pose...
  6% 20/309 [00:09<02:07,  2.27it/s]/content/StridedTransformer-Pose3D/demo/vis.py:224: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig = plt.figure( figsize=(9.6, 5.4))
100% 309/309 [02:24<00:00,  2.14it/s]
Generating 3D pose successful!

Generating demo...
100% 309/309 [02:18<00:00,  2.24it/s]
Generating demo successful!


画像ファイルフォルダをダウンロード

In [9]:
# ダウンロードしたいフォルダを zip 圧縮する  #/content/StridedTransformer-Pose3D/demo/output/IMG_3912/pose2D
!zip -r /content/file.zip /content/StridedTransformer-Pose3D/demo/output/ch/pose2D

# 圧縮した zip ファイルをダウンロードする
from google.colab import files
files.download("/content/file.zip")


  adding: content/StridedTransformer-Pose3D/demo/output/ch/pose2D/ (stored 0%)
  adding: content/StridedTransformer-Pose3D/demo/output/ch/pose2D/0137_2D.png (deflated 6%)
  adding: content/StridedTransformer-Pose3D/demo/output/ch/pose2D/0288_2D.png (deflated 7%)
  adding: content/StridedTransformer-Pose3D/demo/output/ch/pose2D/0044_2D.png (deflated 5%)
  adding: content/StridedTransformer-Pose3D/demo/output/ch/pose2D/0220_2D.png (deflated 6%)
  adding: content/StridedTransformer-Pose3D/demo/output/ch/pose2D/0296_2D.png (deflated 7%)
  adding: content/StridedTransformer-Pose3D/demo/output/ch/pose2D/0174_2D.png (deflated 5%)
  adding: content/StridedTransformer-Pose3D/demo/output/ch/pose2D/0235_2D.png (deflated 6%)
  adding: content/StridedTransformer-Pose3D/demo/output/ch/pose2D/0000_2D.png (deflated 7%)
  adding: content/StridedTransformer-Pose3D/demo/output/ch/pose2D/0042_2D.png (deflated 5%)
  adding: content/StridedTransformer-Pose3D/demo/output/ch/pose2D/0208_2D.png (deflated 6%)
 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

*# テスト動画のセットアップ

In [ ]:
%cd /content/StridedTransformer-Pose3D/demo/video

video_url = 'https://www.youtube.com/watch?v=VSSPwzSflr0' #@param {type:"string"}

#@markdown 動画の切り抜き範囲(秒)を指定してください。\
#@markdown 30秒以上の場合OOM発生の可能性が高いため注意
start_sec =  1#@param {type:"ｄinteger"}
end_sec =  10#@param {type:"integer"}

(start_pt, end_pt) = (start_sec, end_sec)

/content/StridedTransformer-Pose3D/demo/video


In [ ]:
download_resolution = 360
full_video_path = '/content/StridedTransformer-Pose3D/demo/video/full_video.mp4'
file_name = 'input_clip.mp4'
input_clip_path = '/content/StridedTransformer-Pose3D/demo/video/' + file_name

# 動画ダウンロード
ydl_opts = {'format': f'best[height<={download_resolution}]', 'overwrites': True, 'outtmpl': full_video_path}
with YoutubeDL(ydl_opts) as ydl:
    ydl.download([video_url])

# 指定区間切り抜き
with VideoFileClip(full_video_path) as video:
    subclip = video.subclip(start_pt, end_pt)
    subclip.write_videofile(input_clip_path)

[youtube] Extracting URL: https://www.youtube.com/watch?v=VSSPwzSflr0
[youtube] VSSPwzSflr0: Downloading webpage
[youtube] VSSPwzSflr0: Downloading ios player API JSON
[youtube] VSSPwzSflr0: Downloading web creator player API JSON
[youtube] VSSPwzSflr0: Downloading player 53afa3ce
[youtube] VSSPwzSflr0: Downloading m3u8 information
[info] VSSPwzSflr0: Downloading 1 format(s): 18
[download] Destination: /content/StridedTransformer-Pose3D/demo/video/full_video.mp4
[download] 100% of    4.57MiB in 00:00:02 at 1.54MiB/s   
Moviepy - Building video /content/StridedTransformer-Pose3D/demo/video/input_clip.mp4.
MoviePy - Writing audio in input_clipTEMP_MPY_wvf_snd.mp3


MoviePy - Done.
Moviepy - Writing video /content/StridedTransformer-Pose3D/demo/video/input_clip.mp4



Moviepy - Done !
Moviepy - video ready /content/StridedTransformer-Pose3D/demo/video/input_clip.mp4
